# Fine-tuning LegalBERT for SCOTUS Vote Prediction


In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

DATA_PATH = "data/extracted_labelled.csv"
OUTPUT_DIR = "checkpoints/option1"

MODEL_NAME = "lexlms/legal-longformer-base"
MAX_LENGTH = 1024
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5
PATIENCE = 3
SEED = 12345

## 2. Imports

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback,)
from sklearn.metrics import (accuracy_score, f1_score, classification_report, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Torch version: 2.8.0+cu128
CUDA available: True


## 3. Loading Data

In [4]:
df = pd.read_csv(DATA_PATH)
df = df[df["label"].isin([0, 1])].copy()
df["all_utterances"] = df["all_utterances"].fillna("")
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year", "label"]).copy()
df["year"] = df["year"].astype(int)

print(f"Loaded {len(df):,} valid rows. Year range: {df['year'].min()}–{df['year'].max()}")
df.head(3)

Loaded 43,418 valid rows. Year range: 1955–2019


,case_id,year,court,justice_id,label,win_side,all_utterances,total_words,total_utterances,words_to_side0,words_to_side1,word_ratio_0_to_1,questions_to_side0,questions_to_side1,question_ratio_0_to_1,interruptions,issue_area,decision_direction,maj_votes,min_votes
0,1955_71,1955,Warren Court,j__earl_warren,0,0.0,"[UNKNOWN] Number 71, Lonnie Affronti versus Un...",230,17,152,67,2.235294,2,5,0.333333,0,9.0,1.0,9,0
1,1955_71,1955,Warren Court,j__william_o_douglas,0,0.0,[SIDE1] Consecutive sentences.,2,1,0,2,0.000000,0,0,0.000000,0,9.0,1.0,9,0
2,1955_71,1955,Warren Court,j__felix_frankfurter,0,0.0,[SIDE1] This -- is it not necessary for you to...,1209,49,971,238,4.062762,5,2,1.666667,0,9.0,1.0,9,0


## 4. Time Splitting to avoid data leakage

In [ ]:
train_df = df[df["year"] < 2010].reset_index(drop=True)
test_df  = df[df["year"] >= 2010].reset_index(drop=True)  # includes former val (2010–14) + test (>=2015)

for name, split in [("Train (<2010)", train_df), ("Test (>=2010)", test_df)]:
    vc = split["label"].value_counts().sort_index()
    n  = len(split)
    print(f"{name:<16}: {n:6,} rows | label=0: {vc.get(0,0):,} ({vc.get(0,0)/n*100:.1f}%) | label=1: {vc.get(1,0):,} ({vc.get(1,0)/n*100:.1f}%)")

## 5. Model loading

In [6]:
print(f"Loading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

print("Done. :|")

Loading tokenizer and model: nlpaueb/legal-bert-base-uncased


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Done. :|


## 6. Dataset tokenization

In [ ]:
def strip_unknown_utterances(text: str) -> str:
    #Drop [UNKNOWN]-tagged segments, keeping only [SIDE0] and [SIDE1] speech
    parts = text.split(" ||| ")
    parts = [p for p in parts if not p.startswith("[UNKNOWN]")]
    return " ||| ".join(parts)

class SCOTUSDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=8192):
        texts = [strip_unknown_utterances(t) for t in df["all_utterances"].fillna("").tolist()]
        self.labels = df["label"].tolist()
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        # Longformer requires global attention on the CLS token (position 0) for classification
        global_attention_mask = torch.zeros_like(item["attention_mask"])
        global_attention_mask[0] = 1
        item["global_attention_mask"] = global_attention_mask
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("Tokenizing :/")

train_dataset = SCOTUSDataset(train_df, tokenizer, MAX_LENGTH)
test_dataset  = SCOTUSDataset(test_df,  tokenizer, MAX_LENGTH)
print("Done.")

## 7. Training

In [8]:
# weighting for class imbalance
_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].tolist(),
)
class_weights_tensor = torch.tensor(_class_weights, dtype=torch.float32)
print(f"Class weights : class 0 (respondent): {_class_weights[0]:.4f} , class 1 (petitioner): {_class_weights[1]:.4f}")


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights_tensor.to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

Class weights : class 0 (respondent): 1.2640 , class 1 (petitioner): 0.8272


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
    return {
        "accuracy": acc,
        "f1": macro_f1,
        "f1_class0": float(per_class_f1[0]) if len(per_class_f1) > 0 else 0.0,
        "f1_class1": float(per_class_f1[1]) if len(per_class_f1) > 1 else 0.0,
    }

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=50,
    fp16=use_fp16,
    bf16=use_bf16,
    seed=12345,
    report_to="none",
    dataloader_num_workers=0,
    save_total_limit=1,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training :D")
trainer.train()

## 8. Test set eval

In [10]:
test_output = trainer.predict(test_dataset)
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_df["label"].tolist()
probs = torch.softmax(torch.tensor(test_output.predictions, dtype=torch.float32), dim=-1).numpy()

print(classification_report(test_labels, test_preds, target_names=["Respondent (0)", "Petitioner (1)"], digits=4,))

macro_f1 = f1_score(test_labels, test_preds, average="macro", zero_division=0)
acc       = accuracy_score(test_labels, test_preds)

print(f"Overall:  accuracy: {acc:.4f},    macro-F1: {macro_f1:.4f}")

                precision    recall  f1-score   support

Respondent (0)     0.5357    0.4871    0.5103       893
Petitioner (1)     0.6912    0.7311    0.7106      1402

      accuracy                         0.6362      2295
     macro avg     0.6134    0.6091    0.6104      2295
  weighted avg     0.6307    0.6362    0.6326      2295

Overall:  accuracy: 0.6362,    macro-F1: 0.6104


## 9. Storing predictions and best model

In [11]:
pred_df = test_df[["case_id", "justice_id", "year", "label"]].copy()
pred_df["pred_label"] = test_preds
pred_df["prob_class0"] = probs[:, 0]
pred_df["prob_class1"] = probs[:, 1]
pred_df["correct"] = (pred_df["pred_label"] == pred_df["label"]).astype(int)

pred_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
pred_df.to_csv(pred_path, index=False)

print(f"Test predictions saved at: {pred_path}")

best_model_dir = os.path.join(OUTPUT_DIR, "best_model")
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

print(f"Best model saved at: {best_model_dir}")

Test predictions saved at: checkpoints/option1-labelled/test_predictions.csv
Best model saved at: checkpoints/option1-labelled/best_model
